# 4.2 针对硬件的代码生成

通过编译器将高层计算图转换为目标硬件的高效代码。不同硬件架构有不同的计算特性和内存层次，需要针对性的代码生成策略。

## 什么是代码生成？

代码生成是编译器将高层的计算图或中间表示（IR）转换为目标硬件可直接执行的低级代码的过程。核心技术包括：
- **循环展开（Loop Unrolling）**：减少循环控制开销，增加指令级并行度
- **循环分块（Tiling）**：将大矩阵运算分解为适配缓存层次的小块
- **自动向量化**：利用SIMD指令同时处理多个数据元素
- **算子搜索（AutoTuning）**：搜索目标硬件上的最优实现配置

## 为什么用代码生成？

1. **硬件特异性**：同一算子在不同硬件上的最优实现完全不同
2. **内存层次利用**：针对缓存大小分块，使数据复用最大化
3. **编译期全局优化**：运行时框架只能调用预编译算子，代码生成可跨算子优化

## 代码生成的数学原理

**循环分块**：矩阵乘法$C = AB$，分块后$C_{m \times n} = \sum_{k} A_{m \times t_k} \cdot B_{t_k \times n}$，分块大小$(t_m, t_n, t_k)$满足：
$$t_m \cdot t_k \leq |L1|, \quad t_k \cdot t_n \leq |L1|, \quad t_m \cdot t_n \leq |RF|$$

**Roofline模型**：算术强度$I = \frac{\text{FLOPs}}{\text{Bytes}}$决定性能上界，代码生成的目标是使$I$超过阈值进入compute-bound区域。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### TVM风格算子搜索

#### 什么是算子搜索？

TVM将算子实现参数化（如分块大小、展开因子、向量化宽度），构成调度空间（Schedule Space），然后在目标硬件上实测每个配置的性能，选择最优配置。

#### 为什么需要搜索？

1. **硬件行为难以建模**：缓存替换策略、预取机制等硬件行为难以精确建模
2. **参数交互复杂**：分块大小、展开因子之间存在复杂的交互效应
3. **硬件差异大**：不同硬件微架构的最优配置差异显著

#### 算子搜索的数学原理

设调度空间为$\mathcal{S} = \{s_1, s_2, \ldots, s_N\}$，每个调度$s_i$对应运行时延迟$T(s_i)$：
$$s^* = \arg\min_{s \in \mathcal{S}} T(s)$$

- **AutoTVM**：使用XGBoost成本模型预测$\hat{T}(s)$，减少实测次数
- **Ansor**：使用蒙特卡洛树搜索（MCTS）在更大的调度空间中搜索
- 搜索空间大小通常为$10^{10}$级别，需要高效的搜索策略

In [ ]:
class MatMulTuningSpace:
    """模拟TVM的矩阵乘法调优空间"""
    def __init__(self, M, N, K):
        self.M = M
        self.N = N
        self.K = K
        self.tile_sizes = [(16, 16), (32, 32), (64, 64), (128, 128)]
        self.unroll_factors = [1, 2, 4, 8]
        self.vector_lengths = [1, 4, 8]

    def enumerate_configs(self):
        configs = []
        for tile in self.tile_sizes:
            for unroll in self.unroll_factors:
                for vec in self.vector_lengths:
                    configs.append({
                        'tile_m': tile[0], 'tile_n': tile[1],
                        'unroll': unroll, 'vector_len': vec
                    })
        return configs

    def estimate_performance(self, config):
        """模拟性能评估（实际TVM会在硬件上实测）"""
        tile_m = config['tile_m']
        tile_n = config['tile_n']
        unroll = config['unroll']
        vec = config['vector_len']

        cache_efficiency = min(1.0, (tile_m * tile_n) / (64 * 64))
        parallelism = min(1.0, unroll / 4)
        vector_efficiency = min(1.0, vec / 8)
        register_pressure = min(1.0, 1.0 / (tile_m * tile_n / 1024))

        score = cache_efficiency * 0.4 + parallelism * 0.2 + vector_efficiency * 0.3 + register_pressure * 0.1
        return score

M, N, K = 512, 512, 512
tuner = MatMulTuningSpace(M, N, K)
configs = tuner.enumerate_configs()

print(f"=== TVM风格算子调优空间 ===")
print(f"矩阵乘法: {M}x{K} @ {K}x{N}")
print(f"候选配置数: {len(configs)}")

results = [(c, tuner.estimate_performance(c)) for c in configs]
results.sort(key=lambda x: x[1], reverse=True)

print(f"\nTop-5配置:")
for config, score in results[:5]:
    print(f"  tile=({config['tile_m']},{config['tile_n']}) unroll={config['unroll']} vec={config['vector_len']} score={score:.4f}")

print(f"\nTVM调优流程:")
print(f"1. 定义算子计算逻辑")
print(f"2. 枚举可能的调度配置(tile/unroll/vectorize)")
print(f"3. 在目标硬件上实测每个配置")
print(f"4. 选择最优配置生成代码")

### ONNX Runtime 量化推理

#### 什么是ONNX Runtime量化推理？

将模型权重和/或激活值从FP32转换为INT8/INT4表示，利用硬件的低精度算术单元加速推理。

#### 为什么用量化推理？

1. **计算吞吐量提升**：INT8 Tensor Core吞吐量是FP32的2-4倍
2. **内存带宽减少4倍**：INT8每元素1字节 vs FP32每元素4字节
3. **缓存有效容量增加4倍**：可容纳更多数据

#### 量化推理的数学原理

均匀量化：$x_{\text{int}} = \text{clamp}(\lfloor x/s \rceil + z, q_{\min}, q_{\max})$
反量化：$x_{\text{fp32}} \approx s \cdot (x_{\text{int}} - z)$
其中$s = \frac{x_{\max} - x_{\min}}{q_{\max} - q_{\min}}$为缩放因子，$z$为零点。

INT8矩阵乘法：$Y_{\text{int32}} = X_{\text{int8}} \cdot W_{\text{int8}}$，然后$Y_{\text{fp32}} = s_x \cdot s_w \cdot Y_{\text{int32}} + b$

In [ ]:
class ONNXStyleQuantizedMatMul(nn.Module):
    """模拟ONNX Runtime INT8量化矩阵乘法"""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_dim, in_dim) * 0.1)
        self.bias = nn.Parameter(torch.randn(out_dim) * 0.01)

    def forward_fp32(self, x):
        return F.linear(x, self.weight, self.bias)

    def forward_int8(self, x):
        """INT8量化前向传播
        ⚠️ 重要说明：本实现为教学演示，量化后仍转为float计算以简化逻辑。
        实际INT8推理应使用硬件原生INT8算术（如NPU/TPU的INT8 Tensor Core），
        此处仅演示量化/反量化的数学流程，不会产生实际INT8加速效果。
        在CPU上此实现反而可能更慢（额外量化/反量化开销）。
        真正的INT8加速需要：1) 硬件INT8支持 2) 专用kernel 3) per-channel量化"""
        w_scale = self.weight.abs().max(dim=1, keepdim=True)[0] / 127.0
        w_int8 = torch.round(self.weight / w_scale.clamp(min=1e-8)).clamp(-128, 127).to(torch.int8)
        x_scale = x.abs().max(dim=-1, keepdim=True)[0] / 127.0
        x_int8 = torch.round(x / x_scale.clamp(min=1e-8)).clamp(-128, 127).to(torch.int8)
        result_int32 = x_int8.float() @ w_int8.float().T
        result_fp32 = result_int32 * (x_scale * w_scale.T)
        return result_fp32 + self.bias

in_dim, out_dim = 256, 128
q_matmul = ONNXStyleQuantizedMatMul(in_dim, out_dim)
x = torch.randn(64, in_dim)

with torch.no_grad():
    out_fp32 = q_matmul.forward_fp32(x)
    out_int8 = q_matmul.forward_int8(x)
    diff = (out_fp32 - out_int8).norm() / out_fp32.norm() * 100

n_iters = 500
with torch.no_grad():
    start = time.time()
    for _ in range(n_iters):
        _ = q_matmul.forward_fp32(x)
    fp32_time = time.time() - start

    start = time.time()
    for _ in range(n_iters):
        _ = q_matmul.forward_int8(x)
    int8_time = time.time() - start

print(f"=== ONNX Runtime风格INT8推理 ===")
print(f"FP32输出范数: {out_fp32.norm():.4f}")
print(f"INT8输出范数: {out_int8.norm():.4f}")
print(f"相对误差: {diff:.4f}%")
print(f"FP32时间: {fp32_time:.4f}s")
print(f"INT8时间: {int8_time:.4f}s")
print(f"\nONNX Runtime优化:")
print(f"1. INT8算子: 利用VNNI/DP4A等硬件指令")
print(f"2. 图级优化: 常量折叠/算子融合/死代码消除")
print(f"3. 执行提供器: CPU/CUDA/TensorRT/OpenVINO")

### 编译框架对比

#### 什么是编译框架？

深度学习编译框架是将训练好的模型转换为目标硬件高效执行代码的工具链。

#### 为什么需要多种编译框架？

1. **硬件生态不同**：NVIDIA/Apple/ARM/RISC-V需要不同的代码生成后端
2. **部署场景不同**：云端推理/端侧推理对编译时间和运行性能的权衡不同
3. **模型类型不同**：CNN/Transformer/扩散模型的最优编译策略不同

#### 核心编译策略对比

| 策略 | 代表框架 | 原理 | 适用场景 |
|------|---------|------|----------|
| 搜索式编译 | TVM/Ansor | 在调度空间中搜索最优kernel配置 | 跨平台、新硬件适配 |
| 多级IR编译 | MLIR/XLA | 逐级lowering + 逐级优化 | TPU/自定义加速器 |
| 模板式编译 | TensorRT | 预定义优化模板 + 层融合 | NVIDIA GPU推理 |
| 追踪式编译 | torch.compile | 运行时追踪 + JIT编译 | PyTorch快速部署 |

In [ ]:
print(f"{'框架':<20} {'核心方法':<25} {'支持硬件':<25} {'特点':<20}")
print("-" * 90)
frameworks = [
    ("TVM/Apache TVM", "AutoTVM/Ansor搜索", "CPU/GPU/NPU", "跨平台，搜索最优kernel"),
    ("MLIR/XLA", "多级IR逐级lowering", "CPU/GPU/TPU", "编译器基础设施"),
    ("TensorRT", "层融合+精度校准", "NVIDIA GPU", "GPU推理极致优化"),
    ("Core ML Tools", "图优化+ANE编译", "Apple Silicon", "Apple生态最优"),
    ("ONNX Runtime", "图优化+量化+EP", "CPU/GPU/NPU", "通用推理引擎"),
    ("torch.compile", "TorchDynamo+Inductor", "CPU/GPU", "PyTorch原生编译"),
]
for name, method, hw, feature in frameworks:
    print(f"{name:<20} {method:<25} {hw:<25} {feature:<20}")

print(f"\n=== 产业级选择建议 ===")
print(f"1. NVIDIA GPU: TensorRT (最成熟，性能最优)")
print(f"2. Apple端侧: Core ML (ANE加速，生态集成)")
print(f"3. 通用CPU/NPU: ONNX Runtime 或 TVM")
print(f"4. PyTorch快速验证: torch.compile (开发效率高)")

### 产业级实战：真实 ONNX 导出 + ONNX Runtime 推理

ONNX 是产业界最通用的模型交换格式。以下展示完整的端到端流程：
PyTorch模型 → ONNX导出 → ONNX Runtime推理 → 性能对比

#### 关键步骤

1. **模型导出**：`torch.onnx.export()` 将PyTorch模型转换为ONNX格式
2. **模型验证**：`onnx.checker.check_model()` 验证导出的ONNX模型是否合法
3. **推理对比**：比较PyTorch和ONNX Runtime的输出差异和推理速度

#### 导出注意事项

- 需要指定`opset_version`（推荐14+），不同opset支持的算子不同
- 动态形状需通过`dynamic_axes`参数指定
- 导出等价性：$\|f_{\text{PyTorch}}(x) - f_{\text{ONNX}}(x)\|_\infty < 10^{-5}$

In [ ]:
import io
import time
import onnx
import onnxruntime as ort
from transformers import GPT2LMHeadModel, GPT2Config

config = GPT2Config(vocab_size=1000, n_positions=64, n_embd=128, n_layer=4, n_head=4)
gpt2_small = GPT2LMHeadModel(config)
gpt2_small.eval()

class GPT2OnnxWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids):
        return self.model(input_ids).logits

wrapper = GPT2OnnxWrapper(gpt2_small)
dummy_ids = torch.randint(0, 1000, (1, 32))

onnx_buffer = io.BytesIO()
torch.onnx.export(
    wrapper,
    dummy_ids,
    onnx_buffer,
    input_names=['input_ids'],
    output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch', 1: 'seq_len'}, 'logits': {0: 'batch', 1: 'seq_len'}},
    opset_version=14,
    do_constant_folding=True,
)
onnx_buffer.seek(0)
onnx_model = onnx.load(onnx_buffer)
onnx.checker.check_model(onnx_model)
onnx_size = len(onnx_buffer.getvalue())

with torch.no_grad():
    pt_output = wrapper(dummy_ids).numpy()

sess = ort.InferenceSession(onnx_buffer.getvalue())
ort_output = sess.run(None, {'input_ids': dummy_ids.numpy()})[0]

max_diff = np.abs(pt_output - ort_output).max()

print(f"=== ONNX导出 + ONNX Runtime推理 ===")
print(f"模型: GPT-2 (4层, 128维, 1000词表)")
print(f"参数量: {sum(p.numel() for p in gpt2_small.parameters()):,}")
print(f"ONNX文件大小: {onnx_size/1024:.1f} KB")
print(f"PyTorch输出shape: {pt_output.shape}")
print(f"ONNX Runtime输出shape: {ort_output.shape}")
print(f"最大数值差异: {max_diff:.8f}")
print(f"数值一致性: {'PASS' if max_diff < 1e-5 else 'FAIL'}")

In [ ]:
print(f"=== 性能对比: PyTorch vs ONNX Runtime ===")

def benchmark_torch(model, input_ids, warmup=10, runs=50):
    with torch.no_grad():
        for _ in range(warmup):
            model(input_ids)
        start = time.perf_counter()
        for _ in range(runs):
            model(input_ids)
        return (time.perf_counter() - start) / runs * 1000

def benchmark_ort(session, input_ids_np, warmup=10, runs=50):
    for _ in range(warmup):
        session.run(None, {'input_ids': input_ids_np})
    start = time.perf_counter()
    for _ in range(runs):
        session.run(None, {'input_ids': input_ids_np})
    return (time.perf_counter() - start) / runs * 1000

batch_sizes = [1, 4, 8]
seq_len = 32

print(f"{'Batch':<8} {'PyTorch(ms)':<14} {'ORT(ms)':<14} {'加速比':<10}")
print("-" * 46)
for bs in batch_sizes:
    ids = torch.randint(0, 1000, (bs, seq_len))
    pt_time = benchmark_torch(wrapper, ids)
    ort_time = benchmark_ort(sess, ids.numpy())
    speedup = pt_time / ort_time
    print(f"{bs:<8} {pt_time:<14.2f} {ort_time:<14.2f} {speedup:<10.2f}x")

print(f"\nONNX Runtime优化选项:")
sess_opts = ort.SessionOptions()
sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_opts.intra_op_num_threads = 4
sess_optimized = ort.InferenceSession(onnx_buffer.getvalue(), sess_opts)

ids_b1 = torch.randint(0, 1000, (1, seq_len))
ort_opt_time = benchmark_ort(sess_optimized, ids_b1.numpy())
print(f"  默认ORT: {benchmark_ort(sess, ids_b1.numpy()):.2f} ms")
print(f"  优化ORT: {ort_opt_time:.2f} ms")

print(f"\n产业界ONNX部署最佳实践:")
print(f"1. 导出: torch.onnx.export + do_constant_folding=True + opset_version=14+")
print(f"2. 验证: onnx.checker.check_model + 数值一致性对比")
print(f"3. 优化: ORT_ENABLE_ALL + 线程数调优 + IO Binding")
print(f"4. 量化: onnxruntime.quantization.quantize_static (INT8)")
print(f"5. 常见问题: 动态shape → dynamic_axes; 自定义算子 → onnx-script注册")

### Triton Kernel 编写：GPU代码生成的实践

#### 什么是Triton？

Triton是OpenAI开发的GPU编程语言，介于CUDA和高级框架之间。TorchInductor后端自动将FX Graph编译为Triton代码，是PyTorch 2.0编译优化的核心。

#### 为什么学习Triton？

1. **理解编译器输出**：torch.compile生成的就是Triton代码，理解Triton有助于调试和优化
2. **自定义算子**：当框架内置算子不够用时，Triton比CUDA更易编写
3. **产业界趋势**：Triton正在成为GPU编程的事实标准（PyTorch/JAX/XLA均支持）

#### Triton vs CUDA

| 特性 | CUDA | Triton |
|------|------|--------|
| 编程模型 | 手动管理线程 | 基于block的编程 |
| 内存管理 | 手动shared memory | 自动缓存管理 |
| 开发效率 | 低 | 高 |
| 性能上限 | 最高 | 接近CUDA |
| 学习曲线 | 陡峭 | 平缓 |

In [ ]:
try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except ImportError:
    HAS_TRITON = False
    print(f"Triton未安装，使用PyTorch模拟演示")

if HAS_TRITON:
    @triton.jit
    def fused_linear_relu_kernel(
        x_ptr, w_ptr, b_ptr, out_ptr,
        N, K,
        BLOCK_SIZE: tl.constexpr,
    ):
        row_idx = tl.program_id(0)
        row_start = row_idx * K
        x_ptrs = x_ptr + row_start + tl.arange(0, BLOCK_SIZE)
        mask = tl.arange(0, BLOCK_SIZE) < K
        x_row = tl.load(x_ptrs, mask=mask, other=0.0)
        out_vals = tl.zeros((1,), dtype=tl.float32)
        for col in range(N):
            w_ptrs = w_ptr + col * K + tl.arange(0, BLOCK_SIZE)
            w_row = tl.load(w_ptrs, mask=mask, other=0.0)
            dot = tl.sum(x_row * w_row)
            bias = tl.load(b_ptr + col)
            val = tl.maximum(dot + bias, 0.0)
            tl.store(out_ptr + row_idx * N + col, val)

    print(f"Triton已安装，fused_linear_relu_kernel已定义")
    print(f"此kernel将Linear+ReLU融合为单个GPU kernel")
else:
    print(f"\n=== Triton Kernel 示例 (伪代码) ===")
    print(f"""
    @triton.jit
    def fused_linear_relu_kernel(
        x_ptr, w_ptr, b_ptr, out_ptr,
        N, K, BLOCK_SIZE: tl.constexpr,
    ):
        # 每个program处理一行输入
        row_idx = tl.program_id(0)
        # 加载输入行
        x_row = tl.load(x_ptr + row_idx * K + tl.arange(0, BLOCK_SIZE))
        # 逐列计算矩阵乘 + bias + ReLU
        for col in range(N):
            w_col = tl.load(w_ptr + col * K + tl.arange(0, BLOCK_SIZE))
            dot = tl.sum(x_row * w_col)
            bias = tl.load(b_ptr + col)
            val = tl.maximum(dot + bias, 0.0)  # ReLU融合
            tl.store(out_ptr + row_idx * N + col, val)
    """)

print(f"\n=== torch.compile生成的Triton代码查看方法 ===")
print(f"import os")
print(f"os.environ['TORCH_LOGS'] = '+inductor'  # 查看Inductor生成的Triton代码")
print(f"os.environ['TORCH_LOGS'] = '+output_code'  # 查看最终生成的代码")
print(f"\n产业界Triton使用:")
print(f"1. torch.compile后端: 自动生成Triton kernel")
print(f"2. 自定义算子: Flash Attention v2使用Triton编写")
print(f"3. 性能调优: 修改BLOCK_SIZE等参数优化kernel")
print(f"4. 安装: pip install triton (需要CUDA GPU)")

### AutoTVM vs AutoScheduler：搜索策略的演进

#### AutoTVM（基于模板的搜索）

需要人工编写调度模板（schedule template），定义搜索空间，然后使用XGBoost成本模型在空间中搜索最优配置。

#### AutoScheduler / Ansor（无模板搜索）

无需人工模板，使用蒙特卡洛树搜索（MCTS）自动生成调度规则，搜索空间更大但更通用。

#### 对比

| 特性 | AutoTVM | AutoScheduler/Ansor |
|------|---------|---------------------|
| 搜索空间 | 人工定义模板 | 自动生成 |
| 搜索策略 | XGBoost成本模型 | MCTS + 进化搜索 |
| 搜索空间大小 | ~10^8 | ~10^10 |
| 人工介入 | 需要编写模板 | 无需 |
| 搜索时间 | 数小时 | 数十小时 |
| 新硬件适配 | 需重写模板 | 自动适配 |
| 最优性能 | 模板好则最优 | 通常更优 |

#### 编译缓存

端侧部署时编译时间很长（数小时），编译缓存机制至关重要：
- 首次编译：搜索最优配置 → 生成代码 → 缓存结果
- 后续部署：直接加载缓存 → 跳过搜索 → 即时部署
- TVM使用`.log`文件缓存AutoTVM结果，Ansor使用数据库缓存